In [ ]:
# JOUR 3 — MODÉLISATION
# ============================================
# 1. IMPORTS
# ============================================
import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler


In [ ]:
# ============================================
# 2. CHARGEMENT DES DONNÉES
# ============================================
df = pd.read_csv("../House_Prices.csv")

print("Shape :", df.shape)
print("SalePrice manquantes :", df["SalePrice"].isna().sum())
print("SalePrice = 0 :", (df["SalePrice"] == 0).sum())


In [ ]:
# ============================================
# 3. PRÉPARATION REPRODUCTIBLE DES DONNÉES
# ============================================
model_df = df.copy()
print("Shape des données de modélisation :", model_df.shape)


In [ ]:
# ============================================
# 4. FEATURE ENGINEERING REPRODUCTIBLE
# ============================================
model_df["TotalSF"] = (
    model_df["TotalBsmtSF"].fillna(0)
    + model_df["1stFlrSF"].fillna(0)
    + model_df["2ndFlrSF"].fillna(0)
)
model_df["TotalBathrooms"] = (
    model_df["FullBath"].fillna(0)
    + 0.5 * model_df["HalfBath"].fillna(0)
    + model_df["BsmtFullBath"].fillna(0)
    + 0.5 * model_df["BsmtHalfBath"].fillna(0)
)
model_df["HouseAge"] = model_df["YrSold"] - model_df["YearBuilt"]
model_df["RemodAge"] = model_df["YrSold"] - model_df["YearRemodAdd"]

engineered_features = ["TotalSF", "TotalBathrooms", "HouseAge", "RemodAge"]
print(model_df[engineered_features].head())


In [ ]:
# ============================================
# 5. SÉPARATION DE X ET y
# ============================================
y = model_df["SalePrice"]
X = model_df.drop(columns=["SalePrice"])
X = X.drop(columns=["Id"])

print("Shape de X :", X.shape)
print("Shape de y :", y.shape)
print("Features créées dans X :", set(engineered_features).issubset(X.columns))


In [ ]:
# ============================================
# 6. TRAIN / TEST SPLIT
# ============================================
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

print("X_train :", X_train.shape)
print("X_test  :", X_test.shape)


In [ ]:
# ============================================
# 7. TYPES DE VARIABLES
# ============================================
numeric_features = X_train.select_dtypes(include=["int64", "float64"]).columns
categorical_features = X_train.select_dtypes(include=["object"]).columns

print("Variables numériques :", len(numeric_features))
print("Variables catégorielles :", len(categorical_features))


In [ ]:
# ============================================
# 8. PREPROCESSING
# ============================================
numeric_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ]
)

categorical_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features)
    ]
)


In [31]:
# ============================================
# 9. MODÈLES DU JOUR 3
# ============================================
models = {
    "Linear Regression": LinearRegression(),
    "Random Forest": RandomForestRegressor(
        n_estimators=200, random_state=42, n_jobs=-1
    ),
    "Gradient Boosting": GradientBoostingRegressor(
        n_estimators=200, learning_rate=0.05, max_depth=3, random_state=42
    ),
}


In [ ]:
# ============================================
# 10. ENTRAÎNEMENT, PRÉDICTIONS ET ÉVALUATION
# ============================================
results = []
fitted_models = {}

for name, model in models.items():
    pipeline = Pipeline(
        steps=[("preprocessor", preprocessor), ("model", model)]
    )
    pipeline.fit(X_train, y_train)
    predictions = pipeline.predict(X_test)

    results.append({
        "Model": name,
        "MAE": mean_absolute_error(y_test, predictions),
        "RMSE": np.sqrt(mean_squared_error(y_test, predictions)),
        "R2": r2_score(y_test, predictions),
    })
    fitted_models[name] = pipeline


In [ ]:
# ============================================
# 11. TABLEAU COMPARATIF
# ============================================
comparison = pd.DataFrame(results).sort_values("RMSE").reset_index(drop=True)
comparison


# 12. Analyse des modèles

Le tableau comparatif doit être lu ainsi :

- une **MAE** faible signifie que l’erreur moyenne de prédiction est faible ;
- une **RMSE** faible signifie que les erreurs importantes sont limitées ;
- un **R²** élevé signifie que le modèle explique une grande partie de la variation de `SalePrice`.

Le meilleur modèle initial est celui qui a la **RMSE la plus faible**, en vérifiant aussi la MAE et le R².

## Linear Regression

**Avantages :**
- modèle simple et rapide ;
- relativement facile à interpréter ;
- bon modèle de référence.

**Limites :**
- représente moins bien les relations complexes ;
- peut être moins performante si les variables interagissent fortement.

## Random Forest

**Avantages :**
- capture les relations non linéaires ;
- prend en compte les interactions entre variables ;
- généralement robuste.

**Limites :**
- moins interprétable qu’une régression linéaire ;
- plus coûteux à entraîner ;
- peut moins bien extrapoler sur de nouveaux cas atypiques.

## Gradient Boosting

**Avantages :**
- corrige progressivement les erreurs des arbres précédents ;
- souvent performant sur les données tabulaires ;
- modélise des relations complexes.

**Limites :**
- sensible aux hyperparamètres ;
- risque de surapprentissage ;
- nécessite une optimisation qui sera réalisée au Jour 4 avec la validation croisée.

## Choix du modèle

Le choix final reposera d’abord sur la **RMSE**, puis sera confirmé avec la **MAE** et le **R²**. Il faudra aussi considérer le compromis entre précision et interprétabilité.

from sklearn.model_selection import (
    GridSearchCV,
    KFold,
    cross_validate,
    train_test_split,
)

# ÉTAPE 5 — VALIDATION CROISÉE

kfold = KFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

scoring = {
    "mae": "neg_mean_absolute_error",
    "rmse": "neg_root_mean_squared_error",
    "r2": "r2",
}

cv_results = []

for name, pipeline in fitted_models.items():
    scores = cross_validate(
        pipeline,
        X_train,
        y_train,
        cv=kfold,
        scoring=scoring,
        n_jobs=-1
    )

    cv_results.append({
        "Model": name,
        "CV MAE": -scores["test_mae"].mean(),
        "CV RMSE": -scores["test_rmse"].mean(),
        "CV R2": scores["test_r2"].mean(),
    })

cv_comparison = (
    pd.DataFrame(cv_results)
    .sort_values("CV RMSE")
    .reset_index(drop=True)
)

cv_comparison

## Interprétation de la validation croisée

Les données d'entraînement sont découpées en 5 folds. À chaque tour, le modèle apprend sur 4 folds et est validé sur le fold restant.

Le modèle avec la plus petite valeur de CV RMSE est le plus précis en moyenne. Le jeu de test n'est pas utilisé ici : il reste réservé à l'évaluation finale.

# OPTIMISATION DU RANDOM FOREST

random_forest_pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("model", RandomForestRegressor(
            random_state=42,
            n_jobs=-1
        ))
    ]
)

param_grid = {
    "model__n_estimators": [150, 250],
    "model__max_depth": [None, 20],
    "model__min_samples_leaf": [1, 2],
}

grid_search = GridSearchCV(
    estimator=random_forest_pipeline,
    param_grid=param_grid,
    scoring="neg_root_mean_squared_error",
    cv=kfold,
    n_jobs=-1,
    refit=True
)

grid_search.fit(X_train, y_train)

print("Meilleurs paramètres :", grid_search.best_params_)
print("Meilleure CV RMSE :", -grid_search.best_score_)

# COMPARAISON AVANT / APRÈS OPTIMISATION

predictions_rf_initial = fitted_models["Random Forest"].predict(X_test)
predictions_rf_optimise = grid_search.predict(X_test)

optimization_comparison = pd.DataFrame([
    {
        "Model": "Random Forest initial",
        "MAE": mean_absolute_error(y_test, predictions_rf_initial),
        "RMSE": np.sqrt(mean_squared_error(y_test, predictions_rf_initial)),
        "R2": r2_score(y_test, predictions_rf_initial),
    },
    {
        "Model": "Random Forest optimisé",
        "MAE": mean_absolute_error(y_test, predictions_rf_optimise),
        "RMSE": np.sqrt(mean_squared_error(y_test, predictions_rf_optimise)),
        "R2": r2_score(y_test, predictions_rf_optimise),
    }
])

optimization_comparison

## Conclusion de l’optimisation

La validation croisée permet de mesurer la stabilité des modèles sur plusieurs sous-échantillons de données.

Le modèle retenu est celui qui possède la plus faible RMSE moyenne en validation croisée, tout en conservant une MAE faible et un R² élevé.

Après GridSearchCV, le Random Forest optimisé est comparé au modèle initial sur le jeu de test. Si la RMSE diminue et le R² augmente, les nouveaux hyperparamètres améliorent les prédictions. Sinon, le modèle initial reste préférable.